# EasyGen Pro — Smart Inpainting Backend
**Runtime → Change runtime type → T4 GPU** before running!

| Mode | Model | Why |
|------|-------|-----|
| 🗑️ Object Removal | **LaMa** | Seamless fill, no prompt needed |
| 🌄 Background Replace | **Dreamshaper 8** | Prompt-driven, photorealistic |
| ✏️ Custom Fill | **Dreamshaper 8** | Full prompt control |

Run cells **1 → 6** then paste the ngrok URL into `app.html` as `BASE_URL_INPAINT`.

In [ ]:
!pip install "numpy==1.24.4" --force-reinstall --no-deps && pip install "diffusers>=0.30" transformers accelerate "opencv-python-headless==4.8.1.78" simple-lama-inpainting xformers flask flask-cors pyngrok pillow --force-reinstall --no-deps 2>&1 | tail -5

  Using cached numpy-1.24.4.tar.gz (10.9 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
# ── CELL 1: Fresh install ──────────────────────────────────────────────

import subprocess, sys

# Uninstall conflicting packages first
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "numpy", "opencv-python", "opencv-python-headless",
                "simple-lama-inpainting"], capture_output=True)

# Install numpy first at correct version — everything else links against this
!pip install -q "numpy==1.24.4"

# Now install everything else
!pip install -q "diffusers>=0.30" transformers accelerate
!pip install -q "opencv-python-headless==4.8.1.78"
!pip install -q simple-lama-inpainting
!pip install -q xformers flask flask-cors pyngrok pillow

# Verify numpy is correct
import numpy as np
print(f"✅ numpy version: {np.__version__}")
print("✅ All packages installed!")
print("⚠️  Go to Runtime → Restart session, then run Cells 2-6")
# ── CELL 1: Verify install ─────────────────────────────────────────────
import numpy as np
import cv2
import torch
print(f"✅ numpy   : {np.__version__}")
print(f"✅ cv2     : {cv2.__version__}")
print(f"✅ torch   : {torch.__version__}")
print(f"✅ GPU     : {torch.cuda.get_device_name(0)}")
print("All good — run Cells 2-6")

  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires opencv-python>=3.4.8.29, which is not installed.
albumentations 2.0.8 requires opencv-python-headless>=4.9.0.80, which is not installed.
albucore 0.0.24 requires opencv-python-headless>=4.9.0.80, which is not installed.
tensorflow 2.19.0 r

In [ ]:
# ── CELL 2: ngrok token ────────────────────────────────────────────────
import os
from pyngrok import ngrok

# Set NGROK_AUTHTOKEN in the notebook environment; never hard-code credentials.
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
print("✅ ngrok token set")

✅ ngrok token set


In [ ]:
# ── CELL 3: Load LaMa — Object Removal ────────────────────────────────
import torch, numpy as np, cv2
from PIL import Image
from simple_lama_inpainting import SimpleLama

print("⏳ Loading LaMa (~200 MB)…")
lama = SimpleLama()
print(f"✅ LaMa ready on {'GPU' if torch.cuda.is_available() else 'CPU'}")

⏳ Loading LaMa (~200 MB)…
✅ LaMa ready on GPU


In [ ]:
# ── CELL 5: Helper functions (LaMa only) ───────────────────────────────
import cv2

def prepare_lama_mask(image_pil, mask_pil, blur_radius=4):
    """Align mask to image size + dilate + threshold for clean LaMa input."""
    mask_aligned = mask_pil.resize(image_pil.size, Image.LANCZOS).convert("L")
    mask_arr = np.array(mask_aligned)
    mask_arr = (mask_arr > 128).astype(np.uint8) * 255

    # Dilate mask 3px — covers edge artifacts
    kernel   = np.ones((5, 5), np.uint8)
    mask_arr = cv2.dilate(mask_arr, kernel, iterations=2)

    # Feather edges for smooth blending
    if blur_radius > 0:
        k = blur_radius * 2 + 1
        mask_arr = cv2.GaussianBlur(mask_arr, (k, k), blur_radius)
        mask_arr = np.clip(mask_arr, 0, 255).astype(np.uint8)

    return Image.fromarray(mask_arr)

print("✅ Helpers ready")

✅ Helpers ready


In [ ]:
# ── CELL 6: Flask server + ngrok ────────────────────────────────────────
import io, threading
from flask import Flask, request, send_file, jsonify
from flask_cors import CORS
from pyngrok import ngrok

app = Flask(__name__)
CORS(app)


@app.route("/health")
def health():
    return jsonify({"status": "ok", "model": "LaMa (all modes)"})


@app.route("/api/inpaint", methods=["POST"])
def inpaint():
    try:
        if "image" not in request.files:
            return jsonify({"error": "No image"}), 400
        if "mask"  not in request.files:
            return jsonify({"error": "No mask"}),  400

        mode      = request.form.get("mode", "remove")
        mask_blur = max(0, min(20, int(request.form.get("mask_blur", 4))))

        image_pil = Image.open(
            io.BytesIO(request.files["image"].read())
        ).convert("RGB")

        mask_pil = Image.open(
            io.BytesIO(request.files["mask"].read())
        ).convert("L")

        print(f"\n🎨 mode={mode} | size={image_pil.size}")

        # Prepare mask based on mode
        if mode == "remove":
            # Stronger dilation for clean object removal
            clean_mask = prepare_lama_mask(image_pil, mask_pil, blur_radius=mask_blur)

        elif mode == "bg":
            # Invert mask — user paints foreground to KEEP,
            # we inpaint everything else (background)
            mask_aligned = mask_pil.resize(image_pil.size, Image.LANCZOS).convert("L")
            mask_arr = np.array(mask_aligned)
            # Invert: white=background to fill, black=subject to keep
            mask_arr = 255 - (mask_arr > 128).astype(np.uint8) * 255
            kernel   = np.ones((3, 3), np.uint8)
            mask_arr = cv2.dilate(mask_arr, kernel, iterations=1)
            if mask_blur > 0:
                k = mask_blur * 2 + 1
                mask_arr = cv2.GaussianBlur(mask_arr, (k, k), mask_blur)
            clean_mask = Image.fromarray(mask_arr.astype(np.uint8))

        else:  # custom
            clean_mask = prepare_lama_mask(image_pil, mask_pil, blur_radius=mask_blur)

        # Run LaMa
        result_pil = lama(image_pil, clean_mask)

        buf = io.BytesIO()
        result_pil.save(buf, format="PNG")
        buf.seek(0)
        print(f"✅ Done | mode={mode}")
        return send_file(buf, mimetype="image/png")

    except Exception as e:
        import traceback; traceback.print_exc()
        return jsonify({"error": str(e)}), 500


threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=5001, debug=False, use_reloader=False),
    daemon=True
).start()

public_url = ngrok.connect(5001).public_url
print("\n" + "="*60)
print("🚀  LaMa Smart Inpainting backend LIVE:")
print(f"    {public_url}")
print("="*60)
print(f'\n👉  app.html: const BASE_URL_INPAINT = "{public_url}";')
print("\n📌  All 3 modes → LaMa (no diffusers, no numpy conflicts)")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.28.0.12:5001
INFO:werkzeug:Press CTRL+C to quit



🚀  LaMa Smart Inpainting backend LIVE:
    https://semifitted-jordyn-blushingly.ngrok-free.dev

👉  app.html: const BASE_URL_INPAINT = "https://semifitted-jordyn-blushingly.ngrok-free.dev";

📌  All 3 modes → LaMa (no diffusers, no numpy conflicts)


In [ ]:
# ── CELL 7: Debug — save what backend receives ──────────────────────────
# This patches the route to save incoming files so we can see them

@app.route("/api/debug", methods=["POST"])
def debug():
    try:
        image_data = request.files.get("image")
        mask_data  = request.files.get("mask")

        if image_data:
            img = Image.open(io.BytesIO(image_data.read())).convert("RGB")
            img.save("/content/debug_image.png")
            print(f"✅ Image received: {img.size} mode={img.mode}")
        else:
            print("❌ No image received")

        if mask_data:
            msk = Image.open(io.BytesIO(mask_data.read())).convert("L")
            msk.save("/content/debug_mask.png")
            arr = np.array(msk)
            white = (arr > 128).sum()
            print(f"✅ Mask received: {msk.size} white_pixels={white}")
        else:
            print("❌ No mask received")

        return jsonify({"image": str(img.size) if image_data else "missing",
                        "mask_white_pixels": int(white) if mask_data else 0})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

print("✅ Debug endpoint ready at /api/debug")

AssertionError: The setup method 'route' can no longer be called on the application. It has already handled its first request, any changes will not be applied consistently.
Make sure all imports, decorators, functions, etc. needed to set up the application are done before running it.